# Train the detector on Kaggle

This notebook is deliberately thin. A run is defined by a config in `configs/` — `train.yaml`
or one rung of `configs/ladder/` — and not by the order these cells happen to be executed in:
the notebook attaches the data, installs the package and calls one command. If you want to
change the schedule, the subset or the anchors, change the config in the repository and commit
it, so that the run stays reproducible from a file rather than from a browser tab.

**Before running:**

1. Add the LS-SSDD-v1.0 dataset as an input. `data.root` in the config points at
   `/kaggle/input/ls-ssdd-v10/LS-SSDD-v1.0-OPEN`; if the attachment lands somewhere else,
   the first cell below prints where it actually is and the config is the thing to correct.
2. Turn the **Internet** switch on, in the session settings. It is needed once, to fetch the
   COCO backbone weights. Without it, set `model.pretrained: false` and expect much less from
   twelve epochs.
3. Choose the **GPU** accelerator.
4. Set `CONFIG`, in the resume cell below, to the config this session trains — the baseline or
   whichever rung of the ladder is next. It is the one place that names the run: the resume
   cell and the training cell both read it, so there is nothing left to edit twice and nothing
   left to disagree.

**To continue an interrupted run**, attach the previous session's *output* as an input dataset
as well. Kaggle wipes `/kaggle/working` between sessions, so the checkpoint has to come back in
through the door it left by; the third cell copies the last one across before training starts.
Nothing else needs saying — the run reads the directory, sees which epochs are already done and
carries on from the next one.

In [ ]:
!ls /kaggle/input
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# torch and torchvision are already on the image, so the detector extra costs nothing here.
!git clone --depth 1 https://github.com/esamoun/dark-vessel-detection.git /kaggle/working/repo
!pip install -q -e "/kaggle/working/repo[detector]"

In [ ]:
# Bring back the last checkpoint of a previous session, if one is attached as an input.
# Kaggle's working directory does not survive a session; the run's resume does, provided the
# file is put back where the config looks for it.
import re
import shutil
from pathlib import Path

from darkvessel.config import load_config

# The one place a run is named. The training cell below reads it too, so editing this line is
# the whole procedure for switching which rung a session trains or resumes — there is no second
# edit to miss.
CONFIG = "/kaggle/working/repo/configs/train.yaml"

out = load_config(Path(CONFIG))["out"]
working, metrics = Path(out["checkpoints"]), Path(out["metrics"])

# Ordered by epoch, not by path. Sorting paths orders them by the name of the dataset they
# came from, so with two previous outputs attached the older run can win — and it wins quietly.
# Globbing on `working.name` rather than a literal "checkpoints" also narrows this to the rung
# CONFIG names: attaching several previous sessions' outputs at once can no longer surface a
# checkpoint from the wrong rung.
attached = sorted(
    Path("/kaggle/input").glob(f"*/{working.name}/epoch-*.pt"),
    key=lambda path: int(re.findall(r"\d+", path.stem)[-1]),
)

if attached:
    latest = attached[-1]
    working.mkdir(parents=True, exist_ok=True)
    shutil.copy2(latest, working / latest.name)
    # The metrics may legitimately be absent: an epoch's weights land before it is scored, so a
    # session killed in between leaves the checkpoint and no journal. The run scores that epoch
    # from its checkpoint on the way past, which is why this is not an error.
    journal = latest.parent.parent / metrics.name
    if journal.exists():
        metrics.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(journal, metrics)
    print(f"resuming from {latest.name}, metrics {'attached' if journal.exists() else 'missing'}")
else:
    print("no checkpoint attached: this is the first session of the run")

In [ ]:
!darkvessel train --config {CONFIG}

When the session ends, **Save Version** so that `/kaggle/working` becomes an output dataset:
the checkpoints and the metrics file `CONFIG`'s `out.metrics` names are what the next session
resumes from, and that file is what the numbers in the README are copied out of. It is plain
JSON and needs neither torch nor a GPU to read.